<a href="https://colab.research.google.com/github/AkshatkMalik/python-data-science-toolkit/blob/main/Assossiation_rules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ASSIGNMENT-10: ASSOCIATION RULES**

**Task 1: Data Preprocessing**

In this task, we will be loading the Online Retail transaction dataset, and prepare it for performing Association Rule Mining. As we know that the market basket data is usually stored in form of comma separated items for each transaction row, we perform the necessary steps to clean the data, remove null values, duplicates and finally convert the data into one-hot encoded boolean matrix form required for implementing Apriori algorithm.

In [6]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

#  Load the dataset (each row contains comma-separated items for a transaction)
retail_df = pd.read_excel('Online retail.xlsx', header=None)

print('--- Basic Data Exploration ---')
print(f'Total Transactions (Rows): {retail_df.shape[0]}')
print('\nFirst 5 Transactions:')
display(retail_df.head())

#  Preprocess: Convert dataframe column of comma-separated strings into a list of lists (transactions)
transactions = retail_df[0].dropna().apply(lambda x: x.split(',')).tolist()

print(f'\nTotal Cleaned Transactions: {len(transactions)}')
print('Example Transaction Items:', transactions[0][:5])

# Transform data into a One-Hot Encoded format using TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print('\nOne-Hot Encoded DataFrame Shape:', df_encoded.shape)
display(df_encoded.head())

--- Basic Data Exploration ---
Total Transactions (Rows): 7501

First 5 Transactions:


,0
0,"shrimp,almonds,avocado,vegetables mix,green gr..."
1,"burgers,meatballs,eggs"
2,chutney
3,"turkey,avocado"
4,"mineral water,milk,energy bar,whole wheat rice..."



Total Cleaned Transactions: 7501
Example Transaction Items: ['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes']

One-Hot Encoded DataFrame Shape: (7501, 120)


,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,False,True,True,False,True,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False


**Task 2: Association Rule Mining (Apriori Algorithm)**

In this task, using the Python programming language (mlxtend module), we perform the task of finding frequent itemsets and association rules:

1. Frequent Itemsets: Find the set of frequently occurring items in the database with a minimum support value (min_support), which can be set to 0.02 (2%).

2. Association Rules: Determine association rules based on the minimum values of Confidence and Lift, which are used to evaluate the quality and significance of the associations between product pairs.

In [7]:
import warnings

# Suppress deprecation warnings to keep notebook output clean
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [8]:
#  Generate frequent itemsets with a minimum support of 2% (0.02)
frequent_itemsets = apriori(df_encoded, min_support=0.02, use_colnames=True)

print(f'Total Frequent Itemsets Found: {len(frequent_itemsets)}')
print('\nTop 5 Frequent Itemsets:')
display(frequent_itemsets.head())

#  Generate association rules using confidence metric (min threshold = 0.2 or 20%)
rules = association_rules(
    frequent_itemsets, metric='confidence', min_threshold=0.2
)

print(f'\nTotal Association Rules Generated: {len(rules)}')
print('\nTop 5 Association Rules:')
display(
    rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head()
)

Total Frequent Itemsets Found: 103

Top 5 Frequent Itemsets:


,support,itemsets
0,0.020397,(almonds)
1,0.033329,(avocado)
2,0.033729,(brownies)
3,0.087188,(burgers)
4,0.030129,(butter)



Total Association Rules Generated: 55

Top 5 Association Rules:


,antecedents,consequents,support,confidence,lift
0,(burgers),(eggs),0.028796,0.330275,1.837830
1,(burgers),(french fries),0.021997,0.252294,1.476173
2,(burgers),(mineral water),0.024397,0.279817,1.173883
3,(burgers),(spaghetti),0.021464,0.246177,1.413918
4,(cake),(mineral water),0.027463,0.338816,1.421397


**Task 3: Analysis and Interpretation**


In this task, we are faced with interpreting the results of association rule learning and understanding the impact that they might have on the retail business.



In [9]:
# Sort rules by Lift in descending order to find the strongest associations
top_rules = rules.sort_values(by='lift', ascending=False).reset_index(drop=True)

print('--- Top 5 Strongest Association Rules (Sorted by Lift) ---')
display(
    top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    .head(5)
)

--- Top 5 Strongest Association Rules (Sorted by Lift) ---


,antecedents,consequents,support,confidence,lift
0,(spaghetti),(ground beef),0.039195,0.225115,2.291162
1,(ground beef),(spaghetti),0.039195,0.398915,2.291162
2,(olive oil),(spaghetti),0.022930,0.348178,1.999758
3,(soup),(mineral water),0.023064,0.456464,1.914955
4,(frozen vegetables),(milk),0.023597,0.247552,1.910382


Customer Purchasing Behavior Description: Some staple items, such as mineral water, bread, or milk, are often purchased with certain groceries that complement them in a single purchase.



Opportunities: For example, a retail store could use these associations to place products closer to each other so that it is easier for a customer to find what they need. Another option is to provide a “buy X, get Y at a discount” marketing campaign during checkout.

**Task 4: Interview Questions & Core Concepts**

1. Explain what is Lift and why is it important in Association Rules?

First of all, what even is Lift? It describes how many times more likely it is for the antecedent ($A$) and consequent ($B$) to occur together compared to if they were statistically independent.
$$\text{Lift}(A \rightarrow B) = \frac{\text{Support}(A \cup B)}{\text{Support}(A) \times \text{Support}(B)}$$

Why important: Confidence can often be misleading when item $B$ was just absurdly popular on its own (bought by every single customer). Lift eliminates this possibility by normalizing out each item's individual popularity. If $\text{Lift}(A \rightarrow B)$ is $>1$ this is a positive association (they complement each other), $1$ is independent, and $<1$ is a negative association (they contradict).

---

2. What is Support and Confidence? How do we calculate them?

* **Support (for an Itemset $A$):** The percentage of transactions that contain itemset $A$. It describes how popular or frequent a certain item or combination of items is within the data set.
$$\text{Support}(A) = \frac{\text{Transactions containing } A}{\text{Total Transactions}}$$

* **Support (for an Association Rule $A \rightarrow B$):** The percentage of transactions that contain both item $A$ and item $B$.
$$\text{Support}(A \rightarrow B) = \frac{\text{Transactions containing both } A \text{ and } B}{\text{Total Transactions}}$$

* **Confidence:** The conditional probability that item $B$ is purchased given that item $A$ is purchased. Essentially, it's a measurement of how reliable or certain we are that someone who buys item $A$ also buys item $B$.
$$\text{Confidence}(A \rightarrow B) = \frac{\text{Support}(A \cup B)}{\text{Support}(A)} = \frac{\text{Transactions containing both } A \text{ and } B}{\text{Transactions containing } A}$$

---

3. What are some limitations or challenges of Association Rule Mining?

* **Combinatorics / Explosion Problem:** For large retail stores with thousands of different products, it's completely unreasonable to expect to even be able to calculate all possible itemsets (the number of which grows exponentially with product variety) due to memory constraints.
* **Spurious / Accidental Rules:** It's totally possible (especially with low support thresholds) that a rule can seem interesting but not actually have any real-world meaningful connection. This is why we need Lift to filter out these "accidental" rules.
* **Rare vs. Common Items:** If you set your support threshold high enough to include interesting infrequent items (like designer wear), it will also include a ton of uninteresting frequent items (like bottled water). Conversely, setting it low enough to avoid those will make it impossible to get any coherent rules about the rare items.